# Merge Transparent Reporting Created Dates

This notebook loads the three transparent-reporting created-date CSVs in this folder, updates the present-link dataset with the retry-recovered row, normalizes timestamp/type columns, reports type-of-reporting statistics, and saves a merged dataset containing only rows with created/reporting dates.

In [1]:
from pathlib import Path

import pandas as pd

In [2]:
DATA_DIR = Path('../../data/rq1/transparent_data')

PRESENT_CSV = DATA_DIR / 'links_in_message_present_created_dates.csv'
RETRY_OK_CSV = DATA_DIR / 'links_in_message_present_unavailable_retry_recovered_ok.csv'
RECOVERED_CSV = DATA_DIR / 'links_in_message_recovered_created_dates.csv'


present_df = pd.read_csv(PRESENT_CSV)
retry_ok_df = pd.read_csv(RETRY_OK_CSV)
recovered_df = pd.read_csv(RECOVERED_CSV)

print('present:', present_df.shape)
print('retry recovered ok:', retry_ok_df.shape)
print('recovered:', recovered_df.shape)

present: (329, 16)
retry recovered ok: (1, 25)
recovered: (142, 24)


## Add Retry-Recovered Row to Present-Link Data

The retry CSV contains one row that was originally unavailable in `links_in_message_present_created_dates.csv`. This cell updates that row in memory before merging. The CSV in this folder has also been updated on disk.

In [3]:
def apply_retry_updates_to_present(present, retry_ok):
    updated = present.copy()
    update_count = 0

    for _, row in retry_ok.iterrows():
        mask = (
            updated['CVE_ID'].eq(row['CVE_ID'])
            & updated['repository'].eq(row['repository'])
            & updated['PATCH'].eq(row['PATCH'])
            & updated['first_link_in_message'].eq(row['first_link_in_message'])
        )

        if mask.sum() == 0:
            new_row = {col: pd.NA for col in updated.columns}
            for col in updated.columns:
                if col in row.index:
                    new_row[col] = row[col]
            new_row['link_created_at'] = row.get('retry_link_created_at', pd.NA)
            new_row['link_created_date_source'] = row.get('retry_link_created_date_source', pd.NA)
            new_row['link_created_date_status'] = row.get('retry_link_created_date_status', pd.NA)
            new_row['link_created_date_reason'] = row.get('retry_link_created_date_reason', pd.NA)
            new_row['created_date_kind'] = row.get('retry_created_date_kind', pd.NA)
            if 'mined_link_type' in updated.columns:
                new_row['mined_link_type'] = row.get('retry_mined_link_type', row.get('mined_link_type', pd.NA))
            updated = pd.concat([updated, pd.DataFrame([new_row])], ignore_index=True)
            update_count += 1
        else:
            idx = updated.index[mask]
            updated.loc[idx, 'link_created_at'] = row['retry_link_created_at']
            updated.loc[idx, 'link_created_date_source'] = row['retry_link_created_date_source']
            updated.loc[idx, 'link_created_date_status'] = row['retry_link_created_date_status']
            updated.loc[idx, 'link_created_date_reason'] = row['retry_link_created_date_reason']
            updated.loc[idx, 'created_date_kind'] = row['retry_created_date_kind']
            if 'mined_link_type' in updated.columns and 'retry_mined_link_type' in row.index:
                updated.loc[idx, 'mined_link_type'] = row['retry_mined_link_type']
            update_count += int(mask.sum())

    return updated, update_count


present_updated_df, retry_update_count = apply_retry_updates_to_present(present_df, retry_ok_df)
print('retry updates applied:', retry_update_count)

retry updates applied: 1


## Normalize Reporting Types

Two type columns are used:
- `type_of_reporting`: detailed type used for direct inspection.
- `type_of_reporting_broad`: broad category for paper-level reporting.

In [4]:
def clean_string(value):
    if pd.isna(value):
        return pd.NA
    value = str(value).strip()
    return value if value else pd.NA


def derive_detailed_type(row):
    existing = clean_string(row.get('type_of_reporting', pd.NA))
    if not pd.isna(existing):
        return existing

    candidates = [
        row.get('recovered_first_link_type', pd.NA),
        row.get('mined_recovered_link_type', pd.NA),
        row.get('retry_mined_link_type', pd.NA),
        row.get('mined_link_type', pd.NA),
        row.get('first_link_type_for_created_date', pd.NA),
    ]
    text = ' '.join(str(x).lower() for x in candidates if not pd.isna(x))

    if 'github pull' in text:
        return 'github pull request'
    if 'github issue' in text:
        return 'github issue'
    if 'apache issue tracker' in text:
        return 'apache issue tracker'
    if 'jira' in text:
        return 'jira issue tracker'
    if 'bugzilla' in text or 'bz.apache' in text:
        return 'apache issue tracker'
    if 'issue tracker' in text:
        return 'issue tracker'
    if 'github advisory' in text:
        return 'github advisory'
    if 'github commit' in text:
        return 'github commit'
    if 'svn' in text or 'source/tracker' in text or 'source link' in text:
        return 'source/tracker link'
    return 'other'


def broaden_type(detailed_type):
    value = str(detailed_type).strip().lower()
    if value == 'github pull request':
        return 'github pull request'
    if value == 'github issue':
        return 'github issue'
    if 'issue tracker' in value or value in {'apache issue tracker', 'jira issue tracker', 'other issue tracker'}:
        return 'issue tracker'
    return 'other'


def normalize_present(df):
    out = pd.DataFrame({
        'CVE_ID': df['CVE_ID'],
        'repository': df['repository'],
        'PATCH': df['PATCH'],
        'Commit Message': df['Commit Message'],
        'Link Presence': df['Link Presence'],
        'reporting_link': df['first_link_in_message'].fillna(df.get('first_link')),
        'report_created_at': df['link_created_at'],
        'report_created_date_status': df['link_created_date_status'],
        'report_created_date_source': df['link_created_date_source'],
        'report_created_date_reason': df['link_created_date_reason'],
        'created_date_kind': df['created_date_kind'],
        'mined_link_type': df['mined_link_type'],
        'source_dataset': 'links_in_message_present_created_dates',
    })
    out['type_of_reporting'] = df.apply(derive_detailed_type, axis=1)
    out['type_of_reporting_broad'] = out['type_of_reporting'].apply(broaden_type)
    return out


def normalize_retry(df):
    out = pd.DataFrame({
        'CVE_ID': df['CVE_ID'],
        'repository': df['repository'],
        'PATCH': df['PATCH'],
        'Commit Message': df['Commit Message'],
        'Link Presence': df['Link Presence'],
        'reporting_link': df['first_link_in_message'],
        'report_created_at': df['retry_link_created_at'],
        'report_created_date_status': df['retry_link_created_date_status'],
        'report_created_date_source': df['retry_link_created_date_source'],
        'report_created_date_reason': df['retry_link_created_date_reason'],
        'created_date_kind': df['retry_created_date_kind'],
        'mined_link_type': df['retry_mined_link_type'],
        'source_dataset': 'links_in_message_present_unavailable_retry_recovered_ok',
    })
    out['type_of_reporting'] = df.apply(derive_detailed_type, axis=1)
    out['type_of_reporting_broad'] = out['type_of_reporting'].apply(broaden_type)
    return out


def normalize_recovered(df):
    out = pd.DataFrame({
        'CVE_ID': df['CVE_ID'],
        'repository': df['repository'],
        'PATCH': df['PATCH'],
        'Commit Message': df['Commit Message'],
        'Link Presence': df['Link Presence'],
        'reporting_link': df['recovered_link'].fillna(df['recovered_first_link']),
        'report_created_at': df['recovered_link_created_at'],
        'report_created_date_status': df['recovered_link_created_date_status'],
        'report_created_date_source': df['recovered_link_created_date_source'],
        'report_created_date_reason': df['recovered_link_created_date_reason'],
        'created_date_kind': df['created_date_kind'],
        'mined_link_type': df['mined_recovered_link_type'],
        'source_dataset': 'links_in_message_recovered_created_dates',
    })
    out['type_of_reporting'] = df.apply(derive_detailed_type, axis=1)
    out['type_of_reporting_broad'] = out['type_of_reporting'].apply(broaden_type)
    return out

In [5]:
present_norm = normalize_present(present_updated_df)
retry_norm = normalize_retry(retry_ok_df)
recovered_norm = normalize_recovered(recovered_df)

merged_all_df = pd.concat([present_norm, retry_norm, recovered_norm], ignore_index=True)

# The retry row is also represented in the updated present dataset. Keep one row per reporting link.
merged_all_df['source_priority'] = merged_all_df['source_dataset'].map({
    'links_in_message_present_created_dates': 1,
    'links_in_message_present_unavailable_retry_recovered_ok': 2,
    'links_in_message_recovered_created_dates': 3,
}).fillna(99)

merge_key = ['CVE_ID', 'repository', 'PATCH', 'reporting_link']
merged_all_df = (
    merged_all_df
    .sort_values(['source_priority'] + merge_key)
    .drop_duplicates(subset=merge_key, keep='first')
    .drop(columns=['source_priority'])
    .reset_index(drop=True)
)

merged_created_only_df = merged_all_df[
    merged_all_df['report_created_date_status'].eq('ok')
    & merged_all_df['report_created_at'].notna()
].copy()

print('merged all rows:', len(merged_all_df))
print('created-date-only rows:', len(merged_created_only_df))

merged all rows: 471
created-date-only rows: 298


## Reporting Type Summary

This section reports only two tables:
1. Type of reporting across all 471 transparent cases.
2. Type of reporting for cases where a created/reporting date was successfully mined. Percentages in both tables use 471 as the denominator.

In [6]:
TOTAL_TRANSPARENT_CASES = len(merged_all_df)

REPORTING_TYPE_ORDER = [
    'github pull request',
    'github issue',
    'apache issue tracker',
    'github commit',
    'github advisory',
    'other',
]


def reporting_type_for_tables(value):
    value = str(value).strip().lower()
    if value in REPORTING_TYPE_ORDER:
        return value
    # Keep the table taxonomy identical to the created-date table.
    # Non-created-date-only labels such as jira issue tracker, other issue tracker,
    # and source/tracker link are folded into other for this reporting summary.
    return 'other'


merged_all_df['type_of_reporting_table'] = merged_all_df['type_of_reporting'].apply(reporting_type_for_tables)

merged_created_only_df = merged_all_df[
    merged_all_df['report_created_date_status'].eq('ok')
    & merged_all_df['report_created_at'].notna()
].copy()


def reporting_type_count_table(df, count_col):
    table = (
        df
        .groupby('type_of_reporting_table', dropna=False)
        .size()
        .reindex(REPORTING_TYPE_ORDER, fill_value=0)
        .rename_axis('type_of_reporting')
        .reset_index(name=count_col)
    )
    table['% of total 471'] = (table[count_col] / TOTAL_TRANSPARENT_CASES * 100).round(2)
    return table


# Table 1: reporting type distribution among all 471 transparent cases.
type_reporting_all_471 = reporting_type_count_table(merged_all_df, 'Count')

# Table 2: reporting type distribution only for cases with mined created dates.
type_reporting_created_dates = reporting_type_count_table(merged_created_only_df, 'Created date count')

print('Table 1: type of reporting among all transparent cases')
print('Total cases:', TOTAL_TRANSPARENT_CASES)
display(type_reporting_all_471)

print('Table 2: type of reporting among cases with created/reporting date')
print('Created-date cases:', len(merged_created_only_df))
display(type_reporting_created_dates)

Table 1: type of reporting among all transparent cases
Total cases: 471


,type_of_reporting,Count,% of total 471
0,github pull request,116,24.63
1,github issue,71,15.07
2,apache issue tracker,192,40.76
3,github commit,25,5.31
4,github advisory,24,5.10
5,other,43,9.13


Table 2: type of reporting among cases with created/reporting date
Created-date cases: 298


,type_of_reporting,Created date count,% of total 471
0,github pull request,116,24.63
1,github issue,71,15.07
2,apache issue tracker,61,12.95
3,github commit,25,5.31
4,github advisory,24,5.10
5,other,1,0.21


## Reporting Type After Lifecycle Filtering

This table starts from the 298 transparent cases with mined report/created dates and applies the same lifecycle eligibility filters used in the adjusted transparent lifecycle analysis: complete Report/Fix/Release/Disclosure dates, `Release Date >= Fix Date`, and the manual removal of `CVE-2013-5679`.


In [ ]:
FIX_RELEASE_PATH = Path('../../data/rq1/external_release_nvd/fix_releases_from_patch_data_local_server.csv')
NVD_PATH = Path('../../data/rq1/external_release_nvd/cve_NVD_disclosure_dates.csv')

fix_release = pd.read_csv(FIX_RELEASE_PATH)
nvd = pd.read_csv(NVD_PATH)

lifecycle_dates = fix_release[
    fix_release['Oldest Tag Date'].astype(str).str.lower().ne('not found')
].copy()
lifecycle_dates['Fix Date'] = pd.to_datetime(lifecycle_dates['Commit Date'], errors='coerce')
lifecycle_dates['Release Date'] = pd.to_datetime(lifecycle_dates['Oldest Tag Date'], errors='coerce')
lifecycle_dates = lifecycle_dates.merge(nvd[['CVE_ID', 'Published Date']], on='CVE_ID', how='left')
lifecycle_dates['Disclosure Date'] = pd.to_datetime(lifecycle_dates['Published Date'], errors='coerce')

created_lifecycle_df = merged_created_only_df.merge(
    lifecycle_dates[['CVE_ID', 'Fix Date', 'Release Date', 'Disclosure Date']],
    on='CVE_ID',
    how='left',
)

created_lifecycle_df['Report Date'] = pd.to_datetime(
    created_lifecycle_df['report_created_at'],
    errors='coerce',
    utc=True,
).dt.tz_convert(None)

missing_lifecycle_df = created_lifecycle_df[
    created_lifecycle_df[['Report Date', 'Fix Date', 'Release Date', 'Disclosure Date']].isna().any(axis=1)
].copy()

complete_lifecycle_df = created_lifecycle_df.dropna(subset=[
    'Report Date',
    'Fix Date',
    'Release Date',
    'Disclosure Date',
]).copy()

release_before_fix_df = complete_lifecycle_df[
    complete_lifecycle_df['Release Date'] < complete_lifecycle_df['Fix Date']
].copy()

lifecycle_eligible_df = complete_lifecycle_df[
    complete_lifecycle_df['Release Date'] >= complete_lifecycle_df['Fix Date']
].copy()

manual_excluded_lifecycle_df = lifecycle_eligible_df[
    lifecycle_eligible_df['CVE_ID'].eq('CVE-2013-5679')
].copy()

final_lifecycle_reporting_type_df = lifecycle_eligible_df[
    ~lifecycle_eligible_df['CVE_ID'].eq('CVE-2013-5679')
].copy()

reporting_type_after_lifecycle_filtering = pd.DataFrame({
    'Type of reporting': REPORTING_TYPE_ORDER,
})

summary_sources = {
    'Reporting date found': merged_created_only_df,
    'Missing release/tag date': missing_lifecycle_df,
    'Release before fix': release_before_fix_df,
    'Manual exclusion': manual_excluded_lifecycle_df,
    'Final lifecycle rows': final_lifecycle_reporting_type_df,
}

for column_name, df in summary_sources.items():
    counts = (
        df['type_of_reporting_table']
        .value_counts()
        .reindex(REPORTING_TYPE_ORDER, fill_value=0)
    )
    reporting_type_after_lifecycle_filtering[column_name] = reporting_type_after_lifecycle_filtering[
        'Type of reporting'
    ].map(counts).astype(int)

reporting_type_after_lifecycle_filtering['Total reduced'] = (
    reporting_type_after_lifecycle_filtering['Missing release/tag date']
    + reporting_type_after_lifecycle_filtering['Release before fix']
    + reporting_type_after_lifecycle_filtering['Manual exclusion']
)

reporting_type_after_lifecycle_filtering = reporting_type_after_lifecycle_filtering[
    [
        'Type of reporting',
        'Reporting date found',
        'Missing release/tag date',
        'Release before fix',
        'Manual exclusion',
        'Total reduced',
        'Final lifecycle rows',
    ]
]

total_row = {'Type of reporting': 'Total'}
for column in reporting_type_after_lifecycle_filtering.columns[1:]:
    total_row[column] = int(reporting_type_after_lifecycle_filtering[column].sum())

reporting_type_after_lifecycle_filtering = pd.concat([
    reporting_type_after_lifecycle_filtering,
    pd.DataFrame([total_row]),
], ignore_index=True)

print('Reporting type counts after applying adjusted transparent lifecycle filters')
print('Report-date cases:', len(merged_created_only_df))
print('Missing release/tag date:', len(missing_lifecycle_df))
print('Release before fix:', len(release_before_fix_df))
print('Manual exclusion:', len(manual_excluded_lifecycle_df))
print('Final lifecycle rows:', len(final_lifecycle_reporting_type_df))

display(reporting_type_after_lifecycle_filtering)

In [7]:
print('No summary CSVs are saved from this notebook.')
print('all transparent reporting cases:', len(merged_all_df))
print('created-date cases:', len(merged_created_only_df))

No summary CSVs are saved from this notebook.
all transparent reporting cases: 471
created-date cases: 298


In [8]:
merged_created_only_df.head(20)

,CVE_ID,repository,PATCH,Commit Message,Link Presence,reporting_link,report_created_at,report_created_date_status,report_created_date_source,report_created_date_reason,created_date_kind,mined_link_type,source_dataset,type_of_reporting,type_of_reporting_broad,type_of_reporting_table
29,CVE-2011-4905,apache/activemq,https://github.com/apache/activemq/commit/3a71...,Fix for https://issues.apache.org/jira/browse/...,contains links,https://issues.apache.org/jira/browse/AMQ-3294,2011-04-22T18:44:00Z,ok,https://issues.apache.org/jira/browse/AMQ-3294,parsed from Jira HTML fallback,created_at,jira issue tracker,links_in_message_present_created_dates,apache issue tracker,issue tracker,apache issue tracker
53,CVE-2012-6092,apache/activemq,https://github.com/apache/activemq/commit/51eb...,https://issues.apache.org/jira/browse/AMQ-4115...,contains links,https://issues.apache.org/jira/browse/AMQ-4115,2012-10-18T10:04:00Z,ok,https://issues.apache.org/jira/browse/AMQ-4115,parsed from Jira HTML fallback,created_at,jira issue tracker,links_in_message_present_created_dates,apache issue tracker,issue tracker,apache issue tracker
58,CVE-2013-1879,apache/activemq,https://github.com/apache/activemq/commit/148c...,https://issues.apache.org/jira/browse/AMQ-4397...,contains links,https://issues.apache.org/jira/browse/AMQ-4397,2013-03-21T10:14:00Z,ok,https://issues.apache.org/jira/browse/AMQ-4397,parsed from Jira HTML fallback,created_at,jira issue tracker,links_in_message_present_created_dates,apache issue tracker,issue tracker,apache issue tracker
59,CVE-2013-1880,apache/activemq,https://github.com/apache/activemq/commit/fafd...,https://issues.apache.org/jira/browse/AMQ-4398...,contains links,https://issues.apache.org/jira/browse/AMQ-4398,2013-03-21T10:27:00Z,ok,https://issues.apache.org/jira/browse/AMQ-4398,parsed from Jira HTML fallback,created_at,jira issue tracker,links_in_message_present_created_dates,apache issue tracker,issue tracker,apache issue tracker
78,CVE-2013-4600,alkacon/opencms-core,https://github.com/alkacon/opencms-core/commit...,Fixed some XSS problems (github issue #173),contains links,https://github.com/alkacon/opencms-core/issues...,2013-06-13T21:26:01Z,ok,https://api.github.com/repos/alkacon/opencms-c...,NaN,created_at,github issue,links_in_message_present_created_dates,github issue,github issue,github issue
79,CVE-2013-5679,ESAPI/esapi-java-legacy,https://github.com/ESAPI/esapi-java-legacy/com...,Fix for Google Issue #306 and changes to addre...,contains links,https://github.com/ESAPI/esapi-java-legacy/iss...,2014-11-13T18:26:41Z,ok,https://api.github.com/repos/ESAPI/esapi-java-...,NaN,created_at,github issue,links_in_message_present_created_dates,github issue,github issue,github issue
80,CVE-2013-5960,ESAPI/esapi-java-legacy,https://github.com/ESAPI/esapi-java-legacy/com...,Close #306. Close #359,contains links,https://github.com/ESAPI/esapi-java-legacy/iss...,2014-11-13T18:26:41Z,ok,https://api.github.com/repos/ESAPI/esapi-java-...,NaN,created_at,github issue,links_in_message_present_created_dates,github issue,github issue,github issue
86,CVE-2013-7397,AsyncHttpClient/async-http-client,https://github.com/AsyncHttpClient/async-http-...,"Introduce acceptAnyCertificate config, default...",contains links,https://github.com/AsyncHttpClient/async-http-...,2014-04-08T00:56:03Z,ok,https://api.github.com/repos/AsyncHttpClient/a...,NaN,created_at,github pull request,links_in_message_present_created_dates,github pull request,github pull request,github pull request
87,CVE-2013-7398,AsyncHttpClient/async-http-client,https://github.com/AsyncHttpClient/async-http-...,"Introduce acceptAnyCertificate config, default...",contains links,https://github.com/AsyncHttpClient/async-http-...,2013-07-29T23:33:22Z,ok,https://api.github.com/repos/AsyncHttpClient/a...,NaN,created_at,github issue,links_in_message_present_created_dates,github issue,github issue,github issue
96,CVE-2014-0114,apache/commons-beanutils,https://github.com/apache/commons-beanuti